In [5]:
# 01 - Bronze: ingesta real de BTC (Binance) + variables macro
import pandas as pd
import ccxt
from pathlib import Path
from datetime import datetime, timezone
import json

# Si ejecutas en Colab, monta Drive y usa tu ruta de proyecto
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    pass

PROJECT_ROOT = Path('/content/drive/MyDrive/01. USFQ/01. Maestría IA/12. Trabajo de Titulación/05. V2')
BRONZE_MARKET = PROJECT_ROOT / 'data/bronze/market_data'
BRONZE_MACRO = PROJECT_ROOT / 'data/bronze/macro_data'
BRONZE_MARKET.mkdir(parents=True, exist_ok=True)
BRONZE_MACRO.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('BRONZE_MARKET:', BRONZE_MARKET)
print('BRONZE_MACRO:', BRONZE_MACRO)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/01. USFQ/01. Maestría IA/12. Trabajo de Titulación/05. V2
BRONZE_MARKET: /content/drive/MyDrive/01. USFQ/01. Maestría IA/12. Trabajo de Titulación/05. V2/data/bronze/market_data
BRONZE_MACRO: /content/drive/MyDrive/01. USFQ/01. Maestría IA/12. Trabajo de Titulación/05. V2/data/bronze/macro_data


In [ ]:
# Utilidades para estandarizar y guardar en Bronze
def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if isinstance(out.columns, pd.MultiIndex):
        out.columns = [
            '_'.join([str(x) for x in col if str(x) != '']).strip('_').lower()
            for col in out.columns.to_list()
        ]
    else:
        out.columns = [str(c).lower() for c in out.columns]
    return out

def standardize_ohlcv(df: pd.DataFrame, source: str, symbol: str, interval: str) -> pd.DataFrame:
    if df.empty:
        return df

    out = _flatten_columns(df)

    # Timestamp robusto: usa columna si existe; caso contrario usa índice
    if 'timestamp' in out.columns:
        ts = pd.to_datetime(out['timestamp'], utc=True, errors='coerce')
    elif 'datetime' in out.columns:
        ts = pd.to_datetime(out['datetime'], utc=True, errors='coerce')
    elif 'date' in out.columns:
        ts = pd.to_datetime(out['date'], utc=True, errors='coerce')
    else:
        ts = pd.to_datetime(out.index, utc=True, errors='coerce')

    ohlcv = pd.DataFrame({
        'timestamp': ts,
        'open': out['open'] if 'open' in out.columns else pd.NA,
        'high': out['high'] if 'high' in out.columns else pd.NA,
        'low': out['low'] if 'low' in out.columns else pd.NA,
        'close': out['close'] if 'close' in out.columns else pd.NA,
        'volume': out['volume'] if 'volume' in out.columns else pd.NA,
    })

    ohlcv['source'] = source
    ohlcv['symbol'] = symbol
    ohlcv['interval'] = interval

    ohlcv = ohlcv.dropna(subset=['timestamp'])
    ohlcv = ohlcv.sort_values('timestamp').drop_duplicates(subset=['timestamp']).reset_index(drop=True)
    return ohlcv

def save_bronze(df: pd.DataFrame, out_dir: Path, base_name: str, metadata: dict):
    csv_path = out_dir / f'{base_name}.csv'
    json_path = out_dir / f'{base_name}.json'
    df.to_csv(csv_path, index=False)
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2, default=str)
    print(f'OK -> {csv_path.name} ({len(df):,} filas)')
    print(f'OK -> {json_path.name}')

EXCHANGE_CANDIDATES = ['binance', 'binanceus', 'bybit', 'okx', 'kraken']
SYMBOL_BY_EXCHANGE = {
    'binance': 'BTC/USDT',
    'binanceus': 'BTC/USDT',
    'bybit': 'BTC/USDT',
    'okx': 'BTC/USDT',
    'kraken': 'BTC/USD',
}

def _build_exchange(exchange_id: str):
    ex_class = getattr(ccxt, exchange_id)
    return ex_class({'enableRateLimit': True, 'timeout': 30000})

def detect_working_exchange(timeframe: str = '1h') -> str:
    for ex_id in EXCHANGE_CANDIDATES:
        try:
            ex = _build_exchange(ex_id)
            test_symbol = SYMBOL_BY_EXCHANGE[ex_id]
            _ = ex.fetch_ohlcv(test_symbol, timeframe=timeframe, limit=2)
            print(f'Exchange disponible: {ex_id} ({test_symbol})')
            return ex_id
        except Exception as e:
            print(f'No disponible {ex_id}: {str(e)[:140]}')

    raise RuntimeError('Ningún exchange disponible desde esta ubicación. Usa VPN o cambia fuente.')

def fetch_exchange_ohlcv(exchange_id: str, timeframe: str, limit: int = 1000):
    ex = _build_exchange(exchange_id)
    symbol = SYMBOL_BY_EXCHANGE[exchange_id]
    rows = ex.fetch_ohlcv(symbol, timeframe=timeframe, limit=limit)
    df = pd.DataFrame(rows, columns=['timestamp_ms', 'open', 'high', 'low', 'close', 'volume'])
    df['timestamp'] = pd.to_datetime(df['timestamp_ms'], unit='ms', utc=True)
    return df[['timestamp', 'open', 'high', 'low', 'close', 'volume']], symbol

In [7]:
# Descarga real de BTC desde exchange (prioridad Binance)
active_exchange = detect_working_exchange(timeframe='1h')
source_name = f'{active_exchange}_ccxt'
symbol = SYMBOL_BY_EXCHANGE[active_exchange]
prefix = active_exchange

configs = [
    {'interval': '1h', 'limit': 1000, 'name': f'{prefix}_btc_1h'},
    {'interval': '4h', 'limit': 1000, 'name': f'{prefix}_btc_4h'},
    {'interval': '1d', 'limit': 1000, 'name': f'{prefix}_btc_1d'},
]

btc_frames = []
utc_now = datetime.now(timezone.utc).isoformat()

for cfg in configs:
    print(f"Descargando {symbol} | exchange={active_exchange} | interval={cfg['interval']} | limit={cfg['limit']}")
    raw, resolved_symbol = fetch_exchange_ohlcv(
        exchange_id=active_exchange,
        timeframe=cfg['interval'],
        limit=cfg['limit'],
    )

    if raw.empty:
        print('Sin datos para', cfg)
        continue

    std = standardize_ohlcv(raw, source=source_name, symbol=resolved_symbol, interval=cfg['interval'])
    btc_frames.append(std)

    metadata = {
        'dataset': cfg['name'],
        'source': source_name,
        'exchange': active_exchange,
        'symbol': resolved_symbol,
        'interval': cfg['interval'],
        'limit': cfg['limit'],
        'downloaded_at_utc': utc_now,
        'rows': int(len(std)),
        'columns': list(std.columns),
    }
    save_bronze(std, BRONZE_MARKET, cfg['name'], metadata)

No disponible binance: binance GET https://api.binance.com/api/v3/exchangeInfo 451  {
  "code": 0,
  "msg": "Service unavailable from a restricted location accordi
Exchange disponible: binanceus (BTC/USDT)
Descargando BTC/USDT | exchange=binanceus | interval=1h | limit=1000
OK -> binanceus_btc_1h.csv (1,000 filas)
OK -> binanceus_btc_1h.json
Descargando BTC/USDT | exchange=binanceus | interval=4h | limit=1000
OK -> binanceus_btc_4h.csv (1,000 filas)
OK -> binanceus_btc_4h.json
Descargando BTC/USDT | exchange=binanceus | interval=1d | limit=1000
OK -> binanceus_btc_1d.csv (1,000 filas)
OK -> binanceus_btc_1d.json


In [8]:
# Dataset combinado BTC (si faltara 4h, se genera desde 1h)
has_4h = any((not df.empty and df['interval'].iloc[0] == '4h') for df in btc_frames)
btc_1h = next((df for df in btc_frames if (not df.empty and df['interval'].iloc[0] == '1h')), None)

if (not has_4h) and (btc_1h is not None) and (not btc_1h.empty):
    s = btc_1h.copy().set_index('timestamp').sort_index()
    ohlc_4h = pd.DataFrame({
        'open': s['open'].resample('4h').first(),
        'high': s['high'].resample('4h').max(),
        'low': s['low'].resample('4h').min(),
        'close': s['close'].resample('4h').last(),
        'volume': s['volume'].resample('4h').sum(),
    }).dropna().reset_index()

    ohlc_4h['source'] = f'{active_exchange}_ccxt_resampled_from_1h'
    ohlc_4h['symbol'] = symbol
    ohlc_4h['interval'] = '4h'
    btc_frames.append(ohlc_4h)

    meta_4h = {
        'dataset': f'{active_exchange}_btc_4h',
        'source': f'{active_exchange}_ccxt_resampled_from_1h',
        'exchange': active_exchange,
        'symbol': symbol,
        'interval': '4h',
        'downloaded_at_utc': datetime.now(timezone.utc).isoformat(),
        'rows': int(len(ohlc_4h)),
        'columns': list(ohlc_4h.columns),
    }
    save_bronze(ohlc_4h, BRONZE_MARKET, f'{active_exchange}_btc_4h', meta_4h)

if btc_frames:
    combined = pd.concat(btc_frames, ignore_index=True)
    combined = combined.sort_values(['interval', 'timestamp']).reset_index(drop=True)
    meta_combined = {
        'dataset': 'btc_combined_data',
        'source': f'{active_exchange}_ccxt',
        'exchange': active_exchange,
        'symbol': symbol,
        'intervals': sorted(combined['interval'].dropna().unique().tolist()),
        'downloaded_at_utc': datetime.now(timezone.utc).isoformat(),
        'rows': int(len(combined)),
        'columns': list(combined.columns),
    }
    save_bronze(combined, BRONZE_MARKET, 'btc_combined_data', meta_combined)

OK -> btc_combined_data.csv (3,000 filas)
OK -> btc_combined_data.json


In [ ]:
# Descarga de variables macro/proxy de mercado (Yahoo Finance)
import yfinance as yf

macro_tickers = {
    'dxy_usd_index': 'DX-Y.NYB',
    'sp500_index': '^GSPC',
    'nasdaq100_index': '^NDX',
    'gold_futures': 'GC=F',
    'oil_wti_futures': 'CL=F',
    'us10y_yield': '^TNX',
}

macro_frames = []
for name, ticker in macro_tickers.items():
    raw = yf.download(
        tickers=ticker,
        period='max',
        interval='1d',
        auto_adjust=False,
        progress=False,
        group_by='column',
        threads=False,
    )

    if raw.empty:
        print(f'Sin datos: {name} ({ticker})')
        continue

    std = standardize_ohlcv(raw, source='yfinance', symbol=ticker, interval='1d')
    std['feature_name'] = name
    macro_frames.append(std)

    meta = {
        'dataset': f'macro_{name}_1d',
        'source': 'yfinance',
        'symbol': ticker,
        'feature_name': name,
        'interval': '1d',
        'downloaded_at_utc': datetime.now(timezone.utc).isoformat(),
        'rows': int(len(std)),
        'columns': list(std.columns),
    }
    save_bronze(std, BRONZE_MACRO, f'macro_{name}_1d', meta)

if macro_frames:
    macro_combined = pd.concat(macro_frames, ignore_index=True)
    macro_combined = macro_combined.sort_values(['feature_name', 'timestamp']).reset_index(drop=True)
    meta_macro_combined = {
        'dataset': 'macro_combined_1d',
        'source': 'yfinance',
        'interval': '1d',
        'features': sorted(macro_combined['feature_name'].unique().tolist()),
        'downloaded_at_utc': datetime.now(timezone.utc).isoformat(),
        'rows': int(len(macro_combined)),
        'columns': list(macro_combined.columns),
    }
    save_bronze(macro_combined, BRONZE_MACRO, 'macro_combined_1d', meta_macro_combined)

KeyError: Index(['timestamp'], dtype='object')

In [ ]:
# Verificación rápida de salida Bronze
market_files = sorted([p.name for p in BRONZE_MARKET.glob('*')])
macro_files = sorted([p.name for p in BRONZE_MACRO.glob('*')])

print('--- BRONZE MARKET ---')
for f in market_files:
    print(f)

print('\n--- BRONZE MACRO ---')
for f in macro_files:
    print(f)

if (BRONZE_MARKET / 'btc_combined_data.csv').exists():
    display(pd.read_csv(BRONZE_MARKET / 'btc_combined_data.csv').head(5))